# 🔤 Notebook 2: Tokenización y Preprocesamiento de Texto

## 🎯 Objetivos de este Notebook

En este notebook aprenderás:
1. ✅ ¿Qué es la **Tokenización**?
2. ✅ **Limpieza de texto** paso a paso (HTML, puntuación, números)
3. ✅ **Stopwords**: ¿Qué son y por qué removerlas?
4. ✅ **Stemming vs Lemmatization**: Diferencias y cuándo usar cada uno
5. ✅ Pipeline completo de preprocesamiento

⏱️ **Tiempo estimado**: 20 minutos

---

## 🔧 Setup Inicial

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN DE RUTAS - Solución robusta para Jupyter
# ============================================================================

def find_project_root():
    """Encuentra la raíz del proyecto buscando config.py y src/"""
    current = Path.cwd()
    
    # Buscar hacia arriba hasta 5 niveles
    for _ in range(5):
        config_exists = (current / 'config.py').exists()
        src_exists = (current / 'src').exists()
        
        if config_exists and src_exists:
            return current
        
        # También verificar si estamos dentro de sentiment-analysis
        if current.name == 'sentiment-analysis' and src_exists:
            return current
            
        current = current.parent
    
    # Si no encuentra, asumir que es el directorio actual
    return Path.cwd()

# Encontrar y configurar la ruta del proyecto
project_root = find_project_root()
print(f"📁 Proyecto encontrado en: {project_root}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Cambiar el working directory
os.chdir(str(project_root))

# Importar módulos
from src import text_preprocessing

# Descargar recursos de NLTK (si no los tienes)
import nltk
print("📦 Descargando recursos de NLTK...")
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("\n✅ Setup completo!")

## 💡 ¿Qué es la Tokenización?

**Tokenización** = Dividir texto en **unidades** (palabras, oraciones, caracteres)

### Ejemplo Simple:

In [ ]:
# Texto de ejemplo
texto = "I love this movie! It's excellent."

# Tokenización simple (split por espacios)
tokens_simple = texto.split()

print("📝 Texto original:")
print(f"   '{texto}'")
print(f"\n🔤 Tokens (palabras):")
print(f"   {tokens_simple}")
print(f"\n📊 Total de tokens: {len(tokens_simple)}")

### Problema con `split()`:

```python
"movie!" ≠ "movie"  # La puntuación los hace diferentes
"It's" → ["It's"]   # No separa contracciones
```

### Solución: Tokenizador de NLTK

In [ ]:
from nltk.tokenize import word_tokenize

# Tokenización con NLTK (más inteligente)
tokens_nltk = word_tokenize(texto)

print("🔤 Tokens con NLTK:")
print(f"   {tokens_nltk}")
print(f"\n💡 Nota: NLTK separa puntuación y contracciones correctamente")

## 🧹 Limpieza de Texto Paso a Paso

Las reviews reales vienen "sucias". Vamos a limpiarlas paso a paso:

In [ ]:
# Review "sucia" de ejemplo (similar a las reales)
review_sucia = "<p>I ABSOLUTELY LOVED this movie!!! 10/10 stars ⭐⭐⭐<br>Check it out at http://example.com</p>"

print("❌ REVIEW ORIGINAL (sucia):")
print("=" * 70)
print(review_sucia)
print("=" * 70)

### Paso 1: Remover HTML

In [ ]:
paso1 = text_preprocessing.remove_html_tags(review_sucia)

print("1️⃣ Después de remover HTML:")
print(f"   '{paso1}'")

### Paso 2: Remover URLs

In [ ]:
paso2 = text_preprocessing.remove_urls(paso1)

print("2️⃣ Después de remover URLs:")
print(f"   '{paso2}'")

### Paso 3: Convertir a minúsculas

In [ ]:
paso3 = text_preprocessing.to_lowercase(paso2)

print("3️⃣ Después de convertir a minúsculas:")
print(f"   '{paso3}'")
print("\n💡 Ahora 'LOVED', 'Loved' y 'loved' son la misma palabra")

### Paso 4: Remover puntuación

In [ ]:
paso4 = text_preprocessing.remove_punctuation(paso3)

print("4️⃣ Después de remover puntuación:")
print(f"   '{paso4}'")

### Paso 5: Remover números

In [ ]:
paso5 = text_preprocessing.remove_numbers(paso4)

print("5️⃣ Después de remover números:")
print(f"   '{paso5}'")

### Paso 6: Remover espacios extras

In [ ]:
paso6 = text_preprocessing.remove_extra_whitespace(paso5)

print("6️⃣ Después de remover espacios extras:")
print(f"   '{paso6}'")

print("\n" + "="*70)
print("✅ RESULTADO FINAL:")
print("="*70)
print(f"   '{paso6}'")
print("="*70)

### Tabla Comparativa de Transformaciones:

In [ ]:
# Crear tabla comparativa
comparacion = pd.DataFrame({
    'Paso': ['Original', '1-Sin HTML', '2-Sin URLs', '3-Minúsculas', '4-Sin puntuación', '5-Sin números', '6-Final'],
    'Longitud': [len(review_sucia), len(paso1), len(paso2), len(paso3), len(paso4), len(paso5), len(paso6)]
})

# Visualizar
plt.figure(figsize=(12, 5))
plt.bar(comparacion['Paso'], comparacion['Longitud'], color='steelblue', alpha=0.7)
plt.xlabel('Paso de Limpieza', fontsize=12)
plt.ylabel('Longitud (caracteres)', fontsize=12)
plt.title('Reducción de Longitud Durante Limpieza', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📉 Reducción total: {len(review_sucia)} → {len(paso6)} caracteres ({(1-len(paso6)/len(review_sucia))*100:.1f}% reducción)")

## ✂️ Stopwords (Palabras Vacías)

**Stopwords** = Palabras muy comunes que NO aportan información sobre el sentimiento:

- Inglés: *"the", "a", "an", "is", "are", "was", "were", "in", "on"*
- Español: *"el", "la", "de", "que", "y", "en", "un", "una"*

### ¿Por qué removerlas?
- ✅ Reducen el vocabulario
- ✅ El modelo se enfoca en palabras importantes
- ⚠️ Cuidado: A veces importan (ej: "not good" vs "good")

In [ ]:
# Ver stopwords en inglés
stopwords_en = text_preprocessing.get_stopwords('english')

print(f"📚 Total de stopwords en inglés: {len(stopwords_en)}")
print(f"\n🔤 Ejemplos de stopwords:")
print(f"   {list(stopwords_en)[:30]}")

### Ejemplo: Remover Stopwords

In [ ]:
# Texto de ejemplo
texto_con_stops = "the movie was really excellent and the acting was superb"

# Tokenizar
tokens = text_preprocessing.tokenize(texto_con_stops)
print("🔤 Tokens ANTES de remover stopwords:")
print(f"   {tokens}")
print(f"   Total: {len(tokens)} palabras")

# Remover stopwords
tokens_sin_stops = text_preprocessing.remove_stopwords(tokens)
print("\n✂️ Tokens DESPUÉS de remover stopwords:")
print(f"   {tokens_sin_stops}")
print(f"   Total: {len(tokens_sin_stops)} palabras")

# Ver qué se removió
removidos = set(tokens) - set(tokens_sin_stops)
print(f"\n🗑️ Stopwords removidas: {removidos}")

## 🌱 Stemming vs Lemmatization

Ambos reducen palabras a su **forma base**, pero de forma diferente:

### 🔪 Stemming (Derivación)
- **Método**: Algoritmo (corta sufijos/prefijos)
- **Resultado**: Raíz (puede no ser palabra real)
- **Velocidad**: Muy rápido

### 📖 Lemmatization (Lematización)
- **Método**: Diccionario (busca forma base)
- **Resultado**: Palabra válida
- **Velocidad**: Más lento

### Tabla Comparativa:

In [ ]:
# Palabras de ejemplo
palabras_ejemplo = ['running', 'runs', 'ran', 'runner', 'movies', 'better', 'best', 'loving', 'loved']

# Aplicar stemming
stems = text_preprocessing.stem_tokens(palabras_ejemplo)

# Aplicar lemmatization
lemmas = text_preprocessing.lemmatize_tokens(palabras_ejemplo)

# Crear tabla comparativa
comparacion = pd.DataFrame({
    'Palabra Original': palabras_ejemplo,
    'Stemming': stems,
    'Lemmatization': lemmas
})

print("📊 STEMMING vs LEMMATIZATION")
print("="*70)
print(comparacion.to_string(index=False))
print("="*70)

print("\n💡 Observaciones:")
print("   • Stemming: 'movies' → 'movi' (no es palabra real)")
print("   • Lemmatization: 'movies' → 'movie' (palabra válida)")
print("   • Stemming: 'better' → 'better' (no cambia)")
print("   • Lemmatization: 'better' → 'good' (forma base correcta)")

### ¿Cuándo usar cada uno?

| Método | Usar cuando... |
|--------|----------------|
| **Stemming** | • Necesitas **velocidad**<br>• Búsqueda de documentos<br>• No importa precisión |
| **Lemmatization** | • Necesitas **precisión**<br>• Análisis de sentimientos<br>• Importa el significado |

## 🔄 Pipeline Completo de Preprocesamiento

Ahora juntemos todos los pasos en una sola función:

In [ ]:
# Review de ejemplo (muy sucia)
review_test = """<p><b>I ABSOLUTELY LOVED this movie!!!</b> 🎬</p>
The acting was superb and the storyline kept me engaged throughout.
I would give it 10/10 stars! ⭐⭐⭐⭐⭐
Check out the trailer at https://example.com/trailer"""

print("❌ REVIEW ORIGINAL:")
print("="*70)
print(review_test)
print("="*70)

# Aplicar pipeline completo
review_procesada = text_preprocessing.preprocess_text(
    review_test,
    remove_html=True,
    remove_url=True,
    remove_punct=True,
    remove_num=True,
    lowercase=True,
    remove_stops=True,
    apply_lemmatization=True
)

print("\n✅ REVIEW PROCESADA:")
print("="*70)
print(review_procesada)
print("="*70)

print(f"\n📊 Estadísticas:")
print(f"   • Longitud original: {len(review_test)} caracteres")
print(f"   • Longitud procesada: {len(review_procesada)} caracteres")
print(f"   • Reducción: {(1 - len(review_procesada)/len(review_test))*100:.1f}%")
print(f"   • Palabras originales: {len(review_test.split())}")
print(f"   • Palabras procesadas: {len(review_procesada.split())}")

## 🎯 Ejercicio Interactivo

**Prueba el preprocesamiento con tu propia review:**

In [ ]:
# 👇 ESCRIBE TU PROPIA REVIEW AQUÍ:
tu_review = "This movie was TERRIBLE!!! Complete waste of $15 and 2 hours of my life. 😠"

# Procesar
resultado = text_preprocessing.preprocess_text(tu_review)

print("📝 Tu review original:")
print(f"   '{tu_review}'")
print("\n✅ Review procesada:")
print(f"   '{resultado}'")

## 📊 Comparación: Con y Sin Preprocesamiento

Veamos cómo el preprocesamiento afecta múltiples reviews:

In [ ]:
# Reviews de ejemplo
reviews = [
    "This movie is AMAZING! Best film I've ever seen!!!",
    "Terrible waste of time. Don't watch it.",
    "The acting was superb, but the plot was confusing.",
    "I LOVED IT! 10/10 would recommend to everyone!"
]

# Crear DataFrame comparativo
df_comparacion = pd.DataFrame({
    'Original': reviews,
    'Procesada': [text_preprocessing.preprocess_text(r) for r in reviews],
    'Palabras_Orig': [len(r.split()) for r in reviews],
    'Palabras_Proc': [len(text_preprocessing.preprocess_text(r).split()) for r in reviews]
})

print("📊 COMPARACIÓN: ANTES vs DESPUÉS")
print("="*100)
for idx, row in df_comparacion.iterrows():
    print(f"\nReview #{idx+1}:")
    print(f"   Original:  '{row['Original']}'")
    print(f"   Procesada: '{row['Procesada']}'")
    print(f"   Palabras:  {row['Palabras_Orig']} → {row['Palabras_Proc']} ({row['Palabras_Proc']/row['Palabras_Orig']*100:.1f}%)")
    print("-"*100)

## 📊 Resumen de lo Aprendido

En este notebook aprendiste:

✅ **Tokenización**: 
- Dividir texto en palabras (tokens)
- NLTK es más inteligente que `split()`

✅ **Limpieza de Texto** (6 pasos):
1. Remover HTML
2. Remover URLs
3. Convertir a minúsculas
4. Remover puntuación
5. Remover números
6. Remover espacios extras

✅ **Stopwords**:
- Palabras comunes sin significado
- 179 stopwords en inglés
- Removemos para reducir vocabulario

✅ **Stemming vs Lemmatization**:
- Stemming: Rápido pero impreciso ("movi")
- Lemmatization: Lento pero preciso ("movie")
- Para sentimientos: **Lemmatization**

✅ **Pipeline Completo**:
- Función `preprocess_text()` hace todo automáticamente
- Reduce ~70% del texto
- Prepara texto para modelos de ML

---

## 🎓 Próximo Paso

En el **Notebook 3** aprenderás:
- 🔢 Convertir texto limpio → números (TF-IDF)
- 📊 ¿Por qué las computadoras solo entienden números?
- 🎯 Feature extraction para modelos clásicos

**¡Nos vemos en el siguiente notebook!** 🚀